In [1]:
import os
import re
import os.path
import numpy as np
from numpy import log, exp, pi
import pandas as pd
import scipy
import random
from scipy.stats import gaussian_kde, loguniform, gamma
from math import lgamma
from tqdm import tqdm
from ast import literal_eval
from glob import glob
from tqdm import tqdm
from itertools import zip_longest
import numpy.ma as ma # for masked arrays
from astropy.table import Table, join
import astropy.coordinates as coord
import astropy.units as u
import gala.dynamics as gd
import gala.potential as gp
from pyia import GaiaData
from astropy.io import fits
from scipy.stats import ks_2samp
from scipy.stats import anderson_ksamp

# these packages are for fitting with numpyro
import numpyro
from numpyro import distributions as dist, infer
import numpyro_ext
import arviz as az
import jax

# these are psps imports
from psps.transit_class import Population, Star, GeneralStar
import psps.simulate_helpers as simulate_helpers
import psps.simulate_transit as simulate_transit
import psps.utils as utils

# plotting imports
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
matplotlib.rcParams.update({'errorbar.capsize': 1})
pylab_params = {'legend.fontsize': 'large',
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large'}
pylab.rcParams.update(pylab_params)

import warnings
warnings.filterwarnings("ignore")

path = '/Users/chrislam/Desktop/psps/' 

# we're gonna need this for reading in the initial Berger+ 2020 data
def literal_eval_w_exceptions(x):
    try:
        return literal_eval(str(x))   
    except Exception as e:
        pass

def cull4d(df, teff, logg, feh, age, age_err1=None, age_err2=None):
    df = df.loc[(df[teff]<7500)&(df[teff]>3900)]
    df = df.loc[(df[logg]<4.7)&(df[logg]>4.)]
    df = df.loc[(df[feh]<0.25)&(df[feh]>-0.25)]
    df = df.loc[(df[age]<14)]
    """
    try:
        df['frac_age_err1'] = df[age_err1]/df[age]
        df['frac_age_err2'] = np.abs(df[age_err2]/df[age])
        print(np.nanmedian(df['frac_age_err1']), np.nanmedian(df['frac_age_err2']))
        #df = df.loc[(df['frac_age_err1']<0.46)&(df['frac_age_err2']<0.38)]
        df = df.loc[(df['frac_age_err1']<0.50)&(df['frac_age_err2']<0.40)]
    except:
        print("wasn't able to apply fractional age error cut")
        pass
    """

    return df
    

/Users/chrislam/Desktop/psps/src/psps/transit_class.py:380: SyntaxWarning: invalid escape sequence '\s'
  limbach = pd.read_csv(path+'data/limbach_cdfs.txt', engine='python', header=0, sep='\s{2,20}') # space-agnostic separator
/Users/chrislam/Desktop/psps/src/psps/transit_class.py:491: SyntaxWarning: invalid escape sequence '\s'
  limbach = pd.read_csv(path+'data/limbach_cdfs.txt', engine='python', header=0, sep='\s{2,20}') # space-agnostic separator


In [2]:
threshold = 11
frac1 = 0.33
frac2 = 0.33

name_thresh = 11
name_f1 = 33
name_f2 = 33
name = 'step_'+str(name_thresh)+'_'+str(name_f1)+'_'+str(name_f2)
#name = 'monotonic_'+str(name_f1)+'_'+str(name_f2) 
#name = 'piecewise_'+str(name_thresh)+'_'+str(name_f1)+'_'+str(name_f2) 

period_grid = np.logspace(np.log10(1), np.log10(40), 10) # formerly up to 300 days, but that was for Lam+Ballard24. Zink+23 did 1-40 days.
radius_grid = np.linspace(1, 4, 10)

height_bins = np.logspace(2, 3, 6) # ah, so the above are the midpoints of the actual bins they used, I guess
height_bin_midpoints = 0.5 * (np.logspace(2,3,6)[1:] + np.logspace(2,3,6)[:-1])

alpha_se = np.random.normal(-1., 0.2)
alpha_sn = np.random.normal(-1.5, 0.1)

for i in range(30):
    kepler_k2_keep_bootstrap_temp = pd.read_csv(path+f'data/populations/pop_{i}.csv')

    # I forgot about stellar radius and mass when I made these
    kepler_k2_keep_bootstrap_temp['stellar_radius'] = np.random.normal(kepler_k2_keep_bootstrap_temp['Rad'], kepler_k2_keep_bootstrap_temp['e_Rad'])
    kepler_k2_keep_bootstrap_temp['stellar_mass'] = np.random.normal(kepler_k2_keep_bootstrap_temp['Mass'], kepler_k2_keep_bootstrap_temp['e_Mass'])

    pop = Population(kepler_k2_keep_bootstrap_temp['age_draw'], threshold, frac1, frac2)
    frac_hosts = pop.galactic_occurrence_step(threshold, frac1, frac2)
    #frac_hosts = pop.galactic_occurrence_monotonic(frac1, frac2)
    #frac_hosts = pop.galactic_occurrence_piecewise(frac1, frac2, threshold)
    intact_fracs = scipy.stats.truncnorm.rvs(0, 1, loc=0.18, scale=0.1, size=len(kepler_k2_keep_bootstrap_temp))  

    # make planetary systems
    star_data = []
    for i in tqdm(range(len(kepler_k2_keep_bootstrap_temp))): 
        star = GeneralStar(kepler_k2_keep_bootstrap_temp['GaiaDR3'][i], kepler_k2_keep_bootstrap_temp['age_draw'][i], kepler_k2_keep_bootstrap_temp['stellar_radius'][i], kepler_k2_keep_bootstrap_temp['stellar_mass'][i], kepler_k2_keep_bootstrap_temp['teff_draw'][i], kepler_k2_keep_bootstrap_temp['CDPP6'][i], kepler_k2_keep_bootstrap_temp['height'][i]*1000, alpha_se, alpha_sn, frac_hosts[i], intact_fracs[i])
        star_update = {
            'GaiaDR3': star.GaiaDR3,
            'age': star.age,
            'stellar_radius': star.stellar_radius,
            'stellar_mass': star.stellar_mass
        }

  0%|          | 35/18319 [00:00<00:30, 592.99it/s]


KeyboardInterrupt: 